In [ ]:
import numpy as np
import random
import pandas as pd

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8" # set before import torch

import torch
torch.use_deterministic_algorithms(True)

def set_seed(seed):
    random.seed(seed)                     # Python RNG
    np.random.seed(seed)                  # NumPy RNG
    torch.manual_seed(seed)               # CPU RNG
    torch.cuda.manual_seed_all(seed)      # All GPU RNGs
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)  # Enforce deterministic ops

set_seed(91)

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import spearmanr, pearsonr
from transformers import AutoTokenizer, AutoModel, DataCollatorWithPadding, Trainer
import torch.nn as nn
from datasets import Dataset
import numpy as np
import wandb

model_name = "facebook/esm2_t12_35M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModel.from_pretrained(model_name)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Let's use", torch.cuda.device_count(), "GPUs!")
!nvidia-smi

class ExpWeightedMSELoss(nn.Module):
    def __init__(self, alpha=1, ignore_index=-100):
        super().__init__()
        self.alpha = alpha
        self.ignore_index = ignore_index

    def forward(self, y_pred, y_true):
        mask = (y_true != self.ignore_index)
        if mask.any():
            y_pred = y_pred[mask]
            y_true = y_true[mask]
            weights = torch.exp(self.alpha * y_true)
#            skipped = (~mask).sum().item()
#            print(f"[Loss] Skipped {skipped} invalid labels")
            return torch.mean(weights * (y_pred - y_true) ** 2)
        else:
            # No valid labels in batch, return safe zero loss
            return (y_pred - y_pred).sum() * 0.0

class AttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, hidden_states):
        # hidden_states: (batch, seq_len, hidden_size)
        attn_weights = torch.softmax(self.attn(hidden_states), dim=1)  # (batch, seq_len, 1)
        pooled = (attn_weights * hidden_states).sum(dim=1)  # (batch, hidden_size)
        return pooled

class AttentionPooling2(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, hidden_states, mask=None, input_ids=None, cls_token_id=None, eos_token_id=None):
        # hidden_states: (batch, seq_len, hidden_size)
        attn_scores = self.attn(hidden_states).squeeze(-1)  # (batch, seq_len)

        if mask is not None:
            # Optionally zero out CLS and EOS tokens by updating the mask
            if input_ids is not None and cls_token_id is not None and eos_token_id is not None:
                cls_mask = (input_ids == cls_token_id)
                eos_mask = (input_ids == eos_token_id)
                ignore_mask = cls_mask | eos_mask
                mask = mask.masked_fill(ignore_mask, 0)

            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = torch.softmax(attn_scores, dim=1)
        pooled = torch.sum(hidden_states * attn_weights.unsqueeze(-1), dim=1)
        return pooled

class ESM2Regressor(nn.Module):
    def __init__(self, base_model, head_type="linear"):
        super().__init__()
        self.esm = base_model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.cls_token_id = self.tokenizer.cls_token_id
        self.eos_token_id = self.tokenizer.eos_token_id
        
        hidden_size = base_model.config.hidden_size
        self.head_type = head_type.lower()
        self.dropout = nn.Dropout(0.1)

        if self.head_type == "linear":
            self.head = nn.Linear(hidden_size, 1)

        elif self.head_type == "mlp_relu":
            self.head = nn.Sequential(
                nn.Linear(hidden_size, hidden_size),
                nn.ReLU(),
#                nn.Dropout(0.1),
                nn.Linear(hidden_size, 1)
            )
            
        elif self.head_type == "dense":
            self.head = nn.Sequential(
                nn.Linear(hidden_size, hidden_size),
                nn.Linear(hidden_size, 1)
            )

        elif self.head_type == "mlp_gelu":
            self.head = nn.Sequential(
                nn.Linear(hidden_size, hidden_size),
                nn.GELU(),
#                nn.Dropout(0.1),
                nn.Linear(hidden_size, 1)
            )

        elif self.head_type == "residual_mlp":
            self.head = nn.Sequential(
                nn.Linear(hidden_size, 1920),
                nn.GELU(),
                nn.Linear(1920, hidden_size),
                nn.GELU(),
                nn.Linear(hidden_size, 1)
            )

        elif self.head_type == "attention_pooling2":
            self.attn_pool2 = AttentionPooling2(hidden_size)
            self.head = nn.Linear(hidden_size, 1)

        elif self.head_type == "transformer_head":
            encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_size, nhead=8, batch_first=True)
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
            self.head = nn.Linear(hidden_size, 1)

        else:
            raise ValueError(f"Unsupported head_type: {self.head_type}")

    def forward(self, input_ids, attention_mask, labels):
        outputs = self.esm(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state  # (batch, seq_len, hidden_size)

        if self.head_type in ["linear", "dense", "mlp_relu", "mlp_gelu", "residual_mlp"]:
            cls_token = hidden[:, 0, :]  # use CLS token
            x = cls_token
#            x = self.dropout(cls_token)

        elif self.head_type == "attention_pooling2":
            x = self.attn_pool2(hidden_states = hidden, 
                               mask=attention_mask, 
                               input_ids=input_ids, 
                               cls_token_id=self.cls_token_id, 
                               eos_token_id=self.eos_token_id)
#            x = self.dropout(x)

        elif self.head_type == "transformer_head":
            transformed = self.transformer(hidden)
            cls_token = transformed[:, 0, :]
            x = cls_token
#            x = self.dropout(cls_token)

        else:
            raise RuntimeError("Unsupported head type in forward pass.")
        
        logits = self.head(x).squeeze(-1)
        loss = None
        if labels is not None:
            if not torch.is_tensor(labels):
                labels = torch.tensor(labels, dtype=torch.float, device=logits.device)
            else:
                labels = labels.float().to(logits.device)      

            loss_fn = nn.MSELoss()
            loss = loss_fn(logits, labels)
        return {"loss": loss, "logits": logits}

    
def get_layerwise_lr_params(model, base_lr=2e-5, decay=0.95):
    no_decay = ["bias", "LayerNorm.weight"]
    grouped_params = []

    encoder_layers = model.esm.encoder.layer
    num_layers = len(encoder_layers)

    for i, layer in enumerate(encoder_layers):
        lr = base_lr * (decay ** (num_layers - i - 1))

        grouped_params.append({
            "params": [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
            "weight_decay": 0.01,
            "lr": lr,
        })
        grouped_params.append({
            "params": [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
            "lr": lr,
        })

    # Add ESM embeddings (optional)
    grouped_params.append({
        "params": model.esm.embeddings.parameters(),
        "lr": base_lr * (decay ** num_layers)
    })

    # Add custom head
    grouped_params.append({
        "params": model.head.parameters(),  # or model.regressor, etc.
        "lr": base_lr,
        "weight_decay": 0.01,
    })

    return grouped_params



class LLRDTrainer(Trainer):
    def create_optimizer(self):
        if self.optimizer is None:
            optimizer_grouped_parameters = get_layerwise_lr_params(self.model, base_lr=self.args.learning_rate)
            self.optimizer = torch.optim.AdamW(
                optimizer_grouped_parameters,
                lr=self.args.learning_rate,
                betas=(0.9, 0.999),
                eps=1e-8,
            )
        return self.optimizer


wandb.login()

def compute_metrics(eval_pred, ignore_index=-100):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = predictions.squeeze()
    labels = labels.squeeze()
    
    predictions = np.array(predictions)
    labels = np.array(labels)

    # Mask: exclude ignored labels
    mask = (labels != ignore_index)
    length = len(labels)
    mask1 = mask[0:length-101]
    predictions1 = predictions[0:length-101][mask1]
    labels1 = labels[0:length-101][mask1]
    
    mask2 = mask[-101:]
    predictions2 = predictions[-101:][mask2]
    labels2 = labels[-101:][mask2]

    labels = labels[mask]
    if len(labels) == 0:
        return {
            "mse1": None,
            "mse2": None,
            "mae2": None,
            "r2_2": None,
            "spearman2": None,
            "pearson2": None,
            "n_valid": 0,
        }
    try:
        spearman_corr, _ = spearmanr(labels2, predictions2)
    except:
        spearman_corr = None
    try:
        pearson_corr, _ = pearsonr(labels2, predictions2)
    except:
        pearson_corr = None
    mse1 = mean_squared_error(labels1, predictions1)
    mse2 = mean_squared_error(labels2, predictions2)
    mae2 = mean_absolute_error(labels2, predictions2)
    rmse2 = np.sqrt(mse2)
    r2_2 = r2_score(labels2, predictions2)
    return {
        "mse1": mse1,
        "mse2": mse2,
        "mae2": mae2,
        "rmse2": rmse2,
        "r2_2": r2_2,
        "spearman2": spearman_corr,
        "pearson2": pearson_corr,
        "n_valid": len(labels),
    }

from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoModel
from datasets import load_dataset

head_types = ["linear", "dense", "attention_pooling2"]
results = {}

model_name = "facebook/esm2_t12_35M_UR50D"

for head_type in head_types:
    print(f"\n--- Benchmarking head: 8k LLRD model {head_type} ---")

    # Load tokenizer and base model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    base_model = AutoModel.from_pretrained(model_name)

    # Build model
    model = ESM2Regressor(base_model, head_type=head_type)

    # Training arguments
    training_args = TrainingArguments(
        output_dir=f"./8k_LLRD_head_benchmark2_{head_type}",
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        num_train_epochs=5,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        weight_decay=0.01,
        learning_rate=3e-5,
        metric_for_best_model="mse2",
        greater_is_better=False,
        logging_strategy="steps",
        logging_steps=100,
        report_to="wandb",
    )

    # Trainer
    trainer = LLRDTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    run = wandb.init(
    project=f"8k_LLRD_head_benchmark2_logging",      # your W&B project name
    name=f"8k_LLRD_head_benchmark2_{head_type}_run",     # optional custom run name
    config=training_args.to_dict(),   # logs all training args
    )
    
    # Train and evaluate
    trainer.train()
    wandb.finish()
    # trainer.train(resume_from_checkpoint="esm2_ckpt/checkpoint-1000")
    

